# Notebook used to improve the prompt used for answer evaluation

In [ ]:
import json
import re
import textwrap
from concurrent.futures import ThreadPoolExecutor, as_completed

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from langchain_ollama import ChatOllama
from tqdm import tqdm
from viz.tools import merge_list_results

In [ ]:
def print_wrap(text, width=80):
    wrapped = textwrap.fill(text, width=width)
    print(wrapped)

In [ ]:
# load models
llm_judge = ChatOllama(model="llama3.1:8b", temperature=0, num_ctx=15000)
llm70 = ChatOllama(model="llama3.1:70b", temperature=0, num_ctx=15000)
mistral = ChatOllama(model="mistral-small", temperature=0, num_ctx=15000)

## start by splitting data for correlation analysis

In [ ]:
with open(
    "./data/results/retrieved_qu/base_llama3.1:8b.jsonl", "r"
) as f:
    data = json.load(f)

In [ ]:
data[3]

In [ ]:
np.random.choice(data)

In [ ]:
for sample in data:
    if "rating" in sample:
        # remove rating
        sample.pop("rating")

split for justine, benjamin and me

In [ ]:
sample80 = np.random.choice(data, 80, replace=False)

In [ ]:
dico = {}
dico["all"] = list(sample80[:20])
dico["user"] = list(sample80[20:40])
dico["justine"] = list(sample80[40:60])
dico["benjamin"] = list(sample80[60:80])

In [ ]:
len(sample80)

In [ ]:
with open(
    "./data/intermediate_results/correlation_analysis_data.json", "w"
) as f:
    json.dump(dico, f)

In [ ]:
# load data
with open(
    "./data/intermediate_results/correlation_analysis_data.json", "r"
) as f:
    dico = json.load(f)

In [ ]:
len(data)

### Get examples for few shots

In [ ]:
with open(
    "./data/intermediate_results/correlation_analysis/user_data.json",
    "r",
) as f:
    dico = json.load(f)
i = 17

In [ ]:
ratings_done = []
ratings12345 = []
for QA in dico["user"]:
    if QA["rating"] in ratings_done:
        continue
    else:
        ratings_done.append(QA["rating"])
        ratings12345.append(QA)

In [ ]:
ratings12345[0]["context_few_shots"] = """Art. 1
  1
La circulation des véhicules à moteur sur la route cantonale NG 13 Täsch - Zermatt est soumise aux restrictions fonctionnelles de circulation du présent arrêté.
2
Seuls peuvent emprunter la route cantonale NG 13 Täsch - Zermatt les véhicules et les détenteurs de véhicules mis au bénéfice d'une autorisation spéciale valable.
3
… *
Art. 2
  1
Les véhicules d’un poids d’ensemble supérieur à 3,5 tonnes ainsi que les transports exceptionnels peuvent utiliser la route cantonale NG 13 Täsch - Zermatt uniquement munis d'une autorisation spéciale délivrée par le service en charge de la mobilité. *
a) * …
b) * …
c) * …
2
L'autorisation pour emprunter la route cantonale NG 13 Täsch - Zermatt avec des véhicules à moteur d’un poids d’ensemble inférieur à 3,5 tonnes est délivrée par la Police cantonale. *
3
Le Conseil d'Etat fixe le montant de l'émolument de l'autorisation ainsi que sa durée par un arrêté séparé. *
Art. 3 *"""
ratings12345[1][
    "context_few_shots"
] = """Les indices synthétiques de charges des communes valaisannes, calculés sur la base d’une pondération de 1 pour chaque critère (selon le dernier rapport d’efficacité), sont publiés dans le tableau annexé au présent arrêté.
Art. 9
Montant de la compensation des charges
1
La somme à répartir au titre de la compensation des charges est fixée à 22'206'490 francs.
Art. 10
Répartition par habitant aux communes bénéficiaires du fonds de compensation des charges
1
Le montant reçu par habitant par chaque commune bénéficiaire au titre de la répartition du fonds de compensation des charges est publié dans le tableau annexé au présent arrêté (en francs par habitant et au total pour la commune).
Art. 11
Echéance des paiements et versements
1
Si une commune est contributrice à la péréquation des ressources et bénéficiaire de la compensation des charges et/ou de la compensation pour les cas de rigueur, seul le montant net total lui sera facturé ou versé.
"""

ratings12345[2]["context_few_shots"] = """2
La composition de la commission est fixée par voie d'ordonnance.
3
Au besoin, la commission peut faire appel à des consultants externes.
4
La définition des processus, l'inventaire des dégâts et la participation financière font l'objet d'une validation par le Conseil d'Etat.
Art. 33
Contributions financières en faveur des particuliers et des collectivités
1
Sous réserve de la législation spéciale, le Grand Conseil et le Conseil d'Etat peuvent, dans le cadre de leurs compétences respectives, accorder aux particuliers une aide financière pour couvrir les dommages non assurables.
"""
ratings12345[3]["context_few_shots"] = """Art. 56
Résiliation pendant le temps d'essai
1
La résiliation d'un engagement pendant le temps d'essai ne peut intervenir, de part et d'autre, que pour la fin d'un mois, moyennant un préavis de deux semaines.
Art. 57
Résiliation ordinaire par l'employé d'un engagement de durée indéterminée
1
Après le temps d'essai, l'employé peut présenter sa démission moyennant le respect d'un délai de trois mois pour la fin d'un mois."""

ratings12345[4][
    "context_few_shots"
] = """De plus, l'ordonnance énonce les dispositions d'exécution du droit privé fédéral à propos de la procédure préparatoire et de la célébration du mariage. *
1.2.2.2 Fondations
Art. 23
Surveillance des fondations
1
L'organisation de la surveillance des fondations, les modalités de son exercice, ainsi que les émoluments à percevoir font l'objet d'une ordonnance du Conseil d'Etat.
2
Les fondations non encore inscrites au registre du commerce et qui doivent l'être peuvent y être contraintes par l'autorité de surveillance.
3
Le juge de commune avise sans délai l'autorité de surveillance compétente de la création d'une fondation contenue dans une disposition pour cause de mort ouverte par lui.
4
L'autorité de surveillance compétente prend les mesures prévues par l'article 89b CC pour pallier le défaut d'administration de fonds recueillis publiquement. *
Art. 24
Devoir de renseigner"""

In [ ]:
ratings12345[2]

In [ ]:
ratings12345[4]

In [ ]:
# format examples for few shots prompting
def format_examples(dico):
    format = """Question: {question}
Context: {context}
Reference answer: {true_answer}
Generated answer: {generated_answer}
Total Rating: {rating}"""
    string = ""
    for QA in dico:
        question = QA["question"]
        context_few_shots = QA["context_few_shots"]
        true_answer = QA["answer"]
        generated_answer = QA["RAG_answer"]
        string += format.format(
            question=question,
            context=context_few_shots,
            true_answer=true_answer,
            generated_answer=generated_answer,
            rating=QA["rating"],
        )
        string += "\n"
    return string

In [ ]:
examples = format_examples(ratings12345)

In [ ]:
with open(
    "./data/intermediate_results/eval/few_shots_examples.txt", "w"
) as f:
    f.write(examples)

In [ ]:
len(examples.split(" "))

In [ ]:
JUDGE_PROMPT = """
You will be given a user question, a context, a reference answer and a generated answer.
Your task is to provide a 'total rating' scoring how well the generated answer answers the user question.
Give your answer as an integer on a scale of 1 to 5, where 1 means that the generated answer is not helpful at all and entirely false,
and 5 means that the answer completely addresses the question and matches the truth.

Here is the scale you should use to build your answer:
1: The generated answer  is terrible: completely irrelevant to the question asked, or false
2: The generated answer is mostly not helpful, there are invented information, meaning not in the context
3: The generated answer is mostly helpful: information given is true or in the context but question is not answered
4: The generated answer is helpful: is true but it lacks some details
5: The generated answer  is excellent: relevant, true, and exhaustive

Provide your feedback as follows, this is very important. The format of your answer must abslutely follow the following template. Do not add anything alse after your rating:
Template:
Feedback:::
Evaluation: your short rationale for the rating, as a text
Total rating: (x/5)

Now here are the question, the context the reference answer and the generated answer.

Question: {question}
Context: {context}
Reference answer: {true_answer}
Generated answer: {generated_answer}

Feedback:::
Total rating: """

In [ ]:
JUDGE_PROMPT_few_shots = """
You will be given a user question, a context, a reference answer and a generated answer.
Your task is to provide a 'total rating' scoring how well the generated answer answers the user question.
Give your answer as an integer on a scale of 1 to 5, where 1 means that the generated answer is not helpful at all and entirely false,
and 5 means that the answer completely addresses the question and matches the truth.

Here is the scale you should use to build your answer:
1: The generated answer  is terrible: completely irrelevant to the question asked, or false
2: The generated answer is mostly not helpful, there are invented information, meaning not in the context
3: The generated answer is mostly helpful: information given is true or in the context but question is not answered
4: The generated answer is helpful: is true but it lacks some details
5: The generated answer  is excellent: relevant, true, and exhaustive

Here are examples:
{examples}

Provide your feedback as follows, this is very important. The format of your answer must abslutely follow the following template. Do not add anything alse after your rating:
Template:
Feedback:::
Evaluation: your short rationale for the rating, as a text
Total rating: (x/5)

Now here are the question, the context the reference answer and the generated answer.

Question: {question}
Context: {context}
Reference answer: {true_answer}
Generated answer: {generated_answer}

Feedback:::
Total rating: """

In [ ]:
def evaluate_sample(sample, llm_judge, few_shots=False, verbose=False):
    """Evaluate a single generated answer."""
    question = sample["question"]
    generated_answer = sample["RAG_answer"]
    context = sample["context"]
    if generated_answer == "Error":
        sample["rating"] = np.nan
        return sample
    true_answer = sample["answer"]
    if not few_shots:
        prompt = JUDGE_PROMPT.format(
            question=question,
            true_answer=true_answer,
            generated_answer=generated_answer,
            context=context,
        )
    else:
        prompt = JUDGE_PROMPT_few_shots.format(
            question=question,
            true_answer=true_answer,
            generated_answer=generated_answer,
            context=context,
            examples=examples,
        )

    feedback = llm_judge.invoke(prompt)
    if verbose:
        print_wrap(feedback.content)
    try:
        pattern = r"Total rating: ?\(?(\d)(?:/5)?\)?"
        match = re.search(pattern, feedback.content)
        if match:
            rating = int(match.group(1))
        else:
            print(feedback.content)
            rating = np.nan
    except Exception as e:
        print(e)
        print("New eroooooooooooooooooooor")

    if few_shots:
        sample["rating_few_shots"] = rating
    else:
        sample["rating"] = rating
    return sample

In [ ]:
def evaluate_generated_answers_parallel(
    generated_answers, llm_judge, num_workers=4, few_shots=False
):
    """Parallel evaluation using ThreadPoolExecutor."""
    results = []

    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        future_to_sample = {
            executor.submit(
                evaluate_sample, sample.copy(), llm_judge, few_shots
            ): sample
            for sample in generated_answers
        }

        for future in tqdm(
            as_completed(future_to_sample),
            total=len(generated_answers),
            desc="Evaluating",
        ):
            results.append(future.result())

    return results

In [ ]:
random_points = np.random.choice(data, 50)
results_1shot = evaluate_generated_answers_parallel(random_points, llm_judge)
results_few_shots = evaluate_generated_answers_parallel(
    random_points, llm_judge, few_shots=True
)

In [ ]:
merged.keys()

### view difference between with and without few shots

In [ ]:
merged = merge_list_results([results_1shot, results_few_shots], ["1shot", "few_shots"])
merged["difference"] = merged["rating_1shot"] - merged["rating_few_shots"]
merged = merged.merge(
    pd.DataFrame(results_1shot)[["question", "id", "answer", "title"]], on="id"
)
plt.hist(merged["difference"], bins=5, rwidth=0.5, align="mid")
plt.show()

In [ ]:
for sample in merged[merged["difference"] != 0].to_dict(orient="records"):
    print(f"id: {sample['id']}")
    print(f"title: {sample['title']}")
    print("Question:")
    print(sample["question"])
    print_wrap(sample["answer"])
    print("RAG answer:")
    print("-" * 80)
    print_wrap(sample["RAG_answer_1shot"])
    print("Rating 1shot:", sample["rating_1shot"])
    print("Rating few shots:", sample["rating_few_shots"])
    print("\n\n")

In [ ]:
id = "677059a6-f749-49fe-8c72-f326917cdc85"
for sample in data:
    if sample["id"] == id:
        break

In [ ]:
evaluate_sample(sample, mistral, verbose=True, few_shots=True)

### Correlation analysis

In [ ]:
with open(
    "./data/intermediate_results/correlation_analysis/user_data.json",
    "r",
) as f:
    dico_user = json.load(f)

In [ ]:
results_mistral_few_shots = evaluate_generated_answers_parallel(
    dico_user["user"], mistral, few_shots=True
)
results_mistral_1shot = evaluate_generated_answers_parallel(
    dico_user["user"], mistral
)
results_judge_llama_few_shots = evaluate_generated_answers_parallel(
    dico_user["user"], llm_judge, few_shots=True
)
results_judge_llama_1shot = evaluate_generated_answers_parallel(
    dico_user["user"], llm_judge
)

In [ ]:
results_judge_llama70_few_shots = evaluate_generated_answers_parallel(
    dico_user["user"], llm70, few_shots=True
)
results_judge_llama70_1shot = evaluate_generated_answers_parallel(
    dico_user["user"], llm70
)

In [ ]:
results_mistral_1shot[12]

In [ ]:
df_mistral_few_shots = pd.DataFrame(results_mistral_few_shots)[
    ["id", "my_rating", "rating_few_shots"]
]
df_mistral_few_shots.rename(
    columns={"rating_few_shots": "mistral_rating_few_shots"}, inplace=True
)
df_mistral_1shot = pd.DataFrame(results_mistral_1shot)[["id", "rating"]]
df_mistral_1shot.rename(columns={"rating": "mistral_rating_1shot"}, inplace=True)
df_judge_llama_few_shots = pd.DataFrame(results_judge_llama_few_shots)[
    ["id", "rating_few_shots"]
]
df_judge_llama_few_shots.rename(
    columns={"rating_few_shots": "judge_llama8b_rating_few_shots"}, inplace=True
)
df_judge_llama_1shot = pd.DataFrame(results_judge_llama_1shot)[["id", "rating"]]
df_judge_llama_1shot.rename(
    columns={"rating": "judge_llama8b_rating_1shot"}, inplace=True
)
df_judge_llama70_few_shots = pd.DataFrame(results_judge_llama70_few_shots)[
    ["id", "rating_few_shots"]
]
df_judge_llama70_few_shots.rename(
    columns={"rating_few_shots": "judge_llama70b_rating_few_shots"}, inplace=True
)
df_judge_llama70_1shot = pd.DataFrame(results_judge_llama70_1shot)[["id", "rating"]]
df_judge_llama70_1shot.rename(
    columns={"rating": "judge_llama70b_rating_1shot"}, inplace=True
)

merged_df = pd.merge(df_mistral_few_shots, df_mistral_1shot, on="id")
merged_df = pd.merge(merged_df, df_judge_llama_few_shots, on="id")
merged_df = pd.merge(merged_df, df_judge_llama_1shot, on="id")
merged_df = pd.merge(merged_df, df_judge_llama70_few_shots, on="id")
merged_df = pd.merge(merged_df, df_judge_llama70_1shot, on="id")

merged_df.set_index("id", inplace=True)
merged_df.corr()

In [ ]:
results_llm

In [ ]:
random_point = np.random.choice(data)
evaluate_sample(random_point, llm_judge)

In [ ]:
evaluate_sample(random_point, llm70)